<div style="font-size:22pt; line-height:25pt; font-weight:bold; text-align:center;">Notebook 3 — Policy Gradient Methods</div>

<div class="alert alert-success">

**Learning outcomes:**
By the end of this notebook you should be able to:
- state the policy gradient theorem and derive the REINFORCE estimator,
- explain the role of baselines and the advantage function in variance reduction,
- implement REINFORCE and Advantage Actor-Critic (A2C),
- describe the key idea behind PPO and why it improves over A2C.
</div>

# Why Policy Gradients?

So far, value-based methods found $q^*$ and extracted $\pi^*$ greedily.
But:
- Discrete actions only (for DQN).
- The policy is implicit — it cannot be smoothly parameterised.

**Policy gradient methods** directly parameterise and optimise the policy $\pi_\theta$.

<div class="alert alert-success">

**Policy optimisation objective:**
$$J(\theta) = \mathbb{E}_{s_0 \sim \rho_0}\left[ v^{\pi_\theta}(s_0) \right]$$

We want to find $\theta^* = \arg\max_\theta J(\theta)$.
</div>

If $J$ is differentiable (w.r.t. $\theta$), we can use **stochastic gradient ascent**:
$$\theta \leftarrow \theta + \alpha\, \tilde{\nabla}_\theta J(\theta).$$

The challenge: computing $\nabla_\theta J$ requires knowing how the distribution over trajectories changes with $\theta$.

# The (Monte Carlo) Policy Gradient Theorem

<div class="alert alert-success">

**Monte Carlo Policy gradient theorem (infinite horizon, $\gamma$-discounted)**:
$$\nabla_\theta J(\theta) = \mathbb{E}_{(s_t, a_t) \sim \pi_\theta}\!\left[ \sum_{t=0}^\infty \gamma^t G_t \nabla_\theta \log \pi_\theta(a_t | s_t) \right]$$

where $G_t = \sum_{t'=t}^\infty \gamma^{t'-t} r(s_{t'}, a_{t'})$ is the return *from* step $t$.
</div>

**Interpretation**: the gradient pushes $\theta$ to increase the log-probability of actions that led to high returns and decrease it for actions with low returns.

<div class="alert alert-warning">

**Derivation sketch through a series of exercices**  
1. (*nabla-log trick*) Show that $\nabla J(\theta) = \mathbb{E}_\tau \left[ G(\tau) \nabla_\theta \log p(\tau|\theta) \right]$.
2. (*probability of a trajectory*) Show that $p(\tau|\theta) = p(s_0) \prod_{t=0}^\infty p(s_{t+1} | s_t, a_t) \pi_\theta(a_t|s_t)$.
3. (*grad-log-prob of a trajectory*)Deduce that $\nabla_\theta \log p(\tau|\theta) = \sum_{t=0}^\infty \nabla_\theta \log \pi_\theta(a_t|s_t)$.
4. Inject the result of (3) back into (1).
5. (*expected grad-log-prob lemma*) Prove that for any distribution $\rho$ and any action-independent baseline $b(s)$, $\mathbb{E}_{\substack{s\sim\rho \\ a\sim \pi}} \left[ b(s)\left[ \nabla_\theta \log \pi_\theta(a|s) \right] \right] = 0$.
6. Split each $G(\tau)$ in the result of (4) into $\sum_{t'=0}^{t-1} \gamma^{t'} r(s_{t'},a_{t'}) + \sum_{t'=t}^\infty \gamma^{t'} r(s_{t'},a_{t'})$ and use the result of (5) to discard the first term and finish the proof.
</div>

<div class="alert alert-warning">

Why is this called "Monte Carlo"?
</div>

There actually is another policy gradient theorem, that has subtle differences and a different proof.
<div class="alert alert-success">

**Policy gradient theorem:**  
$$\nabla_\theta J(\theta) \propto \mathbb{E}_{\substack{s\sim\rho^\pi \\ a\sim \pi}} \left[ q^\pi(s,a) \nabla_\theta \log\pi(a|s)\right]$$

where $\rho^\pi$ is the (improper) state occupation measure of policy $\pi$.
</div>

Both these estimators estimate the same quantity $\nabla J(\theta)$. The PG theorem is a bit beyond the scope of this class but keep in mind these two differences:
- In the MC policy gradient, state-action pairs are sampled by playing the policy. In the policy gradient theorem, they are drawn from $\rho^\pi$.
- The MC policy gradient has a $\gamma^t$ factor, which is absent in the policy gradient theorem.

<div class="alert alert-warning">

Why will policy gradient methods necessarily be *on-policy* (by the way, let's recall the definition of *on-policy*)?
</div>

# REINFORCE

REINFORCE (Williams, 1992) implements the Monte Carlo estimate of the policy gradient:

<div class="alert alert-success">

**REINFORCE algorithm:**
1. Initialise $\theta$ randomly.
2. Repeat:  
   a. Collect $M$ full trajectories $\{(s_t, a_t, r_t)\}$ using $\pi_\theta$.  
   b. For each $(s_t, a_t)$, compute the return $G_t = \sum_{t'\geq t} \gamma^{t'-t} r_{t'}$.  
   c. Compute the gradient estimate: $\hat{g} = \frac{1}{M} \sum_{\tau} \sum_t \gamma^t G_t \nabla_\theta \log \pi_\theta(a_t|s_t)$.  
   d. Update: $\theta \leftarrow \theta + \alpha \hat{g}$.
</div>

In practice, modern auto-diff libraries compute $\hat{g}$ by defining a **pseudo-loss**:
$$\ell(\theta) = -\frac{1}{M} \sum_\tau \sum_t \gamma^t G_t \log \pi_\theta(a_t|s_t)$$
and calling `loss.backward()`.  The negative sign turns gradient *ascent* on $J$ into gradient *descent* on $\ell$.

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
import matplotlib.pyplot as plt
%matplotlib inline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
class PolicyNetwork(nn.Module):
    """Softmax policy for discrete action spaces."""
    def __init__(self, state_dim, n_actions, nb_neurons=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, nb_neurons), nn.ReLU(),
            nn.Linear(nb_neurons, nb_neurons), nn.ReLU(),
            nn.Linear(nb_neurons, n_actions)
        )

    def forward(self, x):
        return F.softmax(self.net(x), dim=-1)

    def sample_action(self, state):
        """Sample an action and return (action, log_prob)."""
        probs = self(torch.FloatTensor(state).to(device))
        dist = Categorical(probs)
        action = dist.sample()
        return action.item(), dist.log_prob(action)

    def log_prob(self, states, actions):
        """Batch log-probs for a batch of (state, action) pairs."""
        probs = self(states)
        dist = Categorical(probs)
        return dist.log_prob(actions)

In [ ]:
from tqdm import trange

def collect_episode(env, policy):
    """Run one episode and return lists of (state, action, reward, log_prob)."""
    states, actions, rewards, log_probs = [], [], [], []
    state, _ = env.reset()
    done = False
    while not done:
        action, lp = policy.sample_action(state)
        next_state, r, done, trunc, _ = env.step(action)
        states.append(state)
        actions.append(action)
        rewards.append(r)
        log_probs.append(lp)
        state = next_state
        if trunc:
            break
    return states, actions, rewards, log_probs

def compute_returns(rewards, gamma):
    """Compute discounted returns G_t from a list of rewards."""
    G, returns = 0.0, []
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    return returns

class ReinforceAgent:
    def __init__(self, env, config):
        state_dim = env.observation_space.shape[0]
        n_actions = env.action_space.n
        self.policy = PolicyNetwork(state_dim, n_actions).to(device)
        self.gamma  = config.get('gamma', 0.99)
        self.M      = config.get('nb_episodes_per_update', 5)
        self.optim  = torch.optim.Adam(self.policy.parameters(),
                                       lr=config.get('lr', 1e-3))

    def train(self, env, n_updates):
        all_returns = []
        for _ in trange(n_updates):
            batch_loss = 0.0
            batch_return = 0.0
            for _ in range(self.M):
                _, _, rewards, log_probs = collect_episode(env, self.policy)
                returns = compute_returns(rewards, self.gamma)
                batch_return += sum(rewards)
                # Pseudo-loss: − ∑_t γ^t G_t log π(a_t|s_t)
                disc = 1.0
                for lp, G_t in zip(log_probs, returns):
                    batch_loss -= disc * G_t * lp
                    disc *= self.gamma
            batch_loss /= self.M
            self.optim.zero_grad()
            batch_loss.backward()
            self.optim.step()
            all_returns.append(batch_return / self.M)
        return all_returns

In [ ]:
env_cp = gym.make("CartPole-v1", render_mode="rgb_array")

config_reinforce = {
    'gamma': 0.99,
    'lr': 5e-3,
    'nb_episodes_per_update': 8,
}

agent_reinforce = ReinforceAgent(env_cp, config_reinforce)
returns_reinforce = agent_reinforce.train(env_cp, n_updates=50)

In [ ]:
def smooth(x, w=10):
    return np.convolve(x, np.ones(w)/w, mode='valid')

plt.figure(figsize=(8, 3))
plt.plot(returns_reinforce, alpha=0.35, label='episode return')
plt.plot(smooth(returns_reinforce), label='10-update moving avg')
plt.xlabel("Gradient update")
plt.ylabel("Average return per episode")
plt.title("REINFORCE on CartPole-v1")
plt.legend()
plt.tight_layout()
plt.show()

## REINFORCE discussion

Suppose we have a single trajectory, 3 time steps long, drawn following $\pi$, with $G_0 = 10$, $\gamma G_1 = 1$, and $\gamma^2 G_2 = 1$.

Let's write $\nabla_i = \nabla_\theta \log \pi(a_i|s_i)$. 
Finite sample Monte Carlo estimator for $\nabla_\theta J(\theta)$ is:
$$10\nabla_0 + 1\nabla_1 + 1\nabla_2.$$

Following this update of $\theta$, we will hopefully see trajectories go through $s_0,a_0$ a lot more often, and through $s_1,a_1$ and $s_2,a_2$ a little bit more often.

Now recall the expected grad-log-prob lemma: we can substract any action-independent *baseline* $b(s)$ from the realization of $G^\pi(s,a)$ and still obtain an unbiased estimate of $\nabla_\theta J(\theta)$, because $\mathbb{E}_{\tau} \left[b(s) \nabla_\theta \log\pi(a|s) \right]=0$:
$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau} \left[ \left( \gamma^t G_t - b(s) \right) \nabla_\theta \log\pi(a|s)\right].$$

Let's add a constant value of $b(s)=-1$ to all discounted returns. Then our finite sample estimate becomes:
$$9\nabla_0.$$
Let's use $b(s)=-10$.  Then our estimated ascent direction is:
$$-9\nabla_1 -9\nabla_2.$$
Conversely, let's add 20 to all discounted returns. Then the estimate gradient is:
$$30 \nabla_0 + 21\nabla_1 + 21\nabla_2.$$

<div class="alert alert-warning">

We have not changed our MDP, nor our policy, but the gradient estimate is drastically different, both in norm and in direction! So what has happened?
</div>

What has happened is that we have a **finite** sample of states and actions. The baseline $b(s)$ does not affect the gradient estimate's average in the limit of **infinite** sampling. But depending on the baseline's value, the finite sample estimator will have more or less **variance**.

In the previous example, if we augment the probability of $a$ in a given $s$, then we need to decrease it for another $a'$. If this other $a'$ has not been sampled, then its $\nabla\theta \log\pi_\theta(a'|s)$ cannot participate in the estimated ascent direction which might be off just because it strongly depends on which sample set was drawn.

Ideally, we would like to find a baseline which minimizes variance of the estimator, so that whatever finite sample set we draw, we have a good chance of having a good quality gradient estimate.

## Baselines and the Advantage Function

<div class="alert alert-success">

**Policy gradient with baseline:**
$$\nabla_\theta J(\theta) = \mathbb{E}\left[ \sum_t \gamma^t (G_t - b(s_t)) \nabla_\theta \log \pi_\theta(a_t|s_t) \right]$$
</div>

A near-optimal baseline is the value function $v^\pi(s_t)$.
Subtracting it gives the **advantage function**:

<div class="alert alert-success">

$$A^\pi(s, a) = q^\pi(s, a) - v^\pi(s)$$

$A^\pi(s, a) > 0$ means action $a$ is better than the policy's action in $s$;
$A^\pi(s, a) < 0$ means it is worse.
</div>

In practice we estimate $A^\pi$ via the **TD error**:
$$\hat{A}_t = r_t + \gamma v_w(s_{t+1}) - v_w(s_t)$$

where $v_w$ is a learned value network.

# Advantage Actor-Critic (A2C)

A2C maintains two networks updated simultaneously:

| Network | Role | Loss |
|---|---|---|
| **Actor** $\pi_\theta$ | Policy | $-\sum_t \hat{A}_t \log \pi_\theta(a_t|s_t)$ |
| **Critic** $v_w$ | Value estimator | $\sum_t (r_t + \gamma v_w(s_{t+1}) - v_w(s_t))^2$ |

An **entropy bonus** $-\beta \mathcal{H}(\pi(\cdot|s_t))$ is added to the actor loss to encourage exploration.

<div class="alert alert-success">

**A2C update loop:**
1. Collect $n$-step rollouts $(s_t, a_t, r_t)$ with the current policy.
2. Compute TD-based advantage estimates $\hat{A}_t$.
3. Update critic to minimise the TD error.
4. Update actor using $\hat{A}_t$ as weights.
</div>

In [ ]:
class ValueNetwork(nn.Module):
    """Scalar state-value estimator."""
    def __init__(self, state_dim, nb_neurons=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, nb_neurons), nn.ReLU(),
            nn.Linear(nb_neurons, nb_neurons), nn.ReLU(),
            nn.Linear(nb_neurons, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

In [ ]:
from torch.distributions import Categorical

class A2CAgent:
    def __init__(self, env, config):
        state_dim = env.observation_space.shape[0]
        n_actions = env.action_space.n
        self.policy  = PolicyNetwork(state_dim, n_actions).to(device)
        self.critic  = ValueNetwork(state_dim).to(device)
        self.gamma   = config.get('gamma', 0.99)
        self.n_steps = config.get('n_steps', 5)
        self.beta    = config.get('entropy_coeff', 0.01)
        self.lr_actor  = config.get('lr_actor',  1e-3)
        self.lr_critic = config.get('lr_critic', 1e-3)
        self.optim_actor  = torch.optim.Adam(self.policy.parameters(),  lr=self.lr_actor)
        self.optim_critic = torch.optim.Adam(self.critic.parameters(), lr=self.lr_critic)

    def _to_tensor(self, x):
        return torch.FloatTensor(np.array(x)).to(device)

    def collect_rollout(self, env, state):
        """Collect n_steps transitions. Returns (states, actions, rewards, next_states, dones, log_probs, entropies)."""
        states, actions, rewards, next_states, dones = [], [], [], [], []
        log_probs, entropies = [], []
        done = False
        for _ in range(self.n_steps):
            s_tensor = self._to_tensor(state).unsqueeze(0)
            probs = self.policy(s_tensor)
            dist = Categorical(probs)
            action = dist.sample()
            next_state, r, done, trunc, _ = env.step(action.item())
            states.append(state)
            actions.append(action.item())
            rewards.append(r)
            next_states.append(next_state)
            dones.append(float(done or trunc))
            log_probs.append(dist.log_prob(action))
            entropies.append(dist.entropy())
            state = next_state
            if done or trunc:
                state, _ = env.reset()
                done = True
                break
        return state, states, actions, rewards, next_states, dones, log_probs, entropies, done

    def update(self, states, actions, rewards, next_states, dones, log_probs, entropies):
        S  = self._to_tensor(states)
        S2 = self._to_tensor(next_states)
        R  = self._to_tensor(rewards)
        D  = self._to_tensor(dones)

        with torch.no_grad():
            V2 = self.critic(S2)
            targets = R + self.gamma * (1 - D) * V2

        V = self.critic(S)
        advantages = (targets - V).detach()

        # Critic update
        critic_loss = F.mse_loss(V, targets)
        self.optim_critic.zero_grad()
        critic_loss.backward()
        self.optim_critic.step()

        # Actor update
        log_p = torch.stack(log_probs)
        ent   = torch.stack(entropies)
        actor_loss = -(advantages * log_p).mean() - self.beta * ent.mean()
        self.optim_actor.zero_grad()
        actor_loss.backward()
        self.optim_actor.step()

        return critic_loss.item(), actor_loss.item()

    def train(self, env, total_steps):
        state, _ = env.reset()
        ep_returns = []
        ep_return  = 0.0
        for _ in trange(total_steps // self.n_steps):
            state, states, actions, rewards, next_states, dones, log_probs, entropies, ep_done = \
                self.collect_rollout(env, state)
            ep_return += sum(rewards)
            self.update(states, actions, rewards, next_states, dones, log_probs, entropies)
            if ep_done:
                ep_returns.append(ep_return)
                ep_return = 0.0
        return ep_returns

In [ ]:
env_cp2 = gym.make("CartPole-v1", render_mode="rgb_array")

config_a2c = {
    'gamma': 0.99,
    'lr_actor':  3e-4,
    'lr_critic': 1e-3,
    'n_steps': 5,
    'entropy_coeff': 0.01,
}

agent_a2c = A2CAgent(env_cp2, config_a2c)
returns_a2c = agent_a2c.train(env_cp2, total_steps=80_000)

In [ ]:
plt.figure(figsize=(9, 3))
if len(returns_a2c) > 10:
    plt.plot(returns_a2c, alpha=0.3, label='episode return')
    plt.plot(smooth(returns_a2c, 20), label='20-ep moving avg')
plt.xlabel("Episode")
plt.ylabel("Return")
plt.title("A2C on CartPole-v1")
plt.legend()
plt.tight_layout()
plt.show()
print(f"Last 50-episode average: {np.mean(returns_a2c[-50:]):.1f}")

## REINFORCE vs A2C comparison

| Property | REINFORCE | A2C |
|---|---|---|
| Gradient estimator | Full Monte Carlo returns | TD(0) with advantage baseline |
| Variance | High | Lower |
| Bias | None (exact MC) | Low (bootstrap from $v_w$) |
| Critic network | No | Yes |
| Sample efficiency | Low | Better |
| Implementation | Simple | Moderate |

A2C is strictly more efficient than REINFORCE thanks to the variance-reducing advantage baseline, at the cost of maintaining and training a value network.

# Opening towards PPO

A2C can take overly large gradient steps that degrade the policy.
**Proximal Policy Optimisation (PPO)** constrains how much the policy changes per update.

**PPO-clip objective:**
$$\mathcal{L}^{\text{clip}}(\theta) = \mathbb{E}_t\!\left[ \min\!\left( r_t(\theta)\, \hat{A}_t,\ \text{clip}(r_t(\theta), 1{-}\epsilon, 1{+}\epsilon)\, \hat{A}_t \right) \right]$$

where $r_t(\theta) = \dfrac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$ is the **probability ratio** between the new and old policy.

The clip prevents $r_t(\theta)$ from going beyond $[1-\epsilon, 1+\epsilon]$, which keeps the update close to the old policy (**trust region**).

PPO is currently one of the most widely used RL algorithms in practice:
- Works for discrete and continuous action spaces.
- Strong performance on a wide range of tasks.
- Available in stable-baselines3: `from stable_baselines3 import PPO`.

<div class="alert alert-warning">

**Think about it:**
Why would taking a very large gradient step in policy space be dangerous?
Hint: the policy gradient is an *on-policy* estimator — the samples were collected with $\pi_{\theta_{\text{old}}}$.
</div>

# Back to V2G

<div class="alert alert-warning">

**Discussion:**
1. Are policy gradient methods better suited to V2G than DQN?
2. The V2G action space is continuous. Which algorithms from this notebook can handle that natively?
3. What architecture would you choose for the actor network in V2G?
   (Hint: the output must lie in $[-1, 1]^{10}$.)
</div>

In [ ]:
# Benchmarking with stable-baselines3 PPO on V2G (if installed)
try:
    from stable_baselines3 import PPO
    from stable_baselines3.common.env_util import make_vec_env
    from ev2gym.models.ev2gym_env import EV2Gym
    from ev2gym.rl_agent.state import V2G_profit_max

    env_v2g = EV2Gym(config_file="custom.yaml", save_replay=False,
                     save_plots=False, state_function=V2G_profit_max)

    print("Training PPO on V2G for 50k steps (≈ a few minutes on CPU)...")
    model = PPO("MlpPolicy", env_v2g, verbose=0,
                learning_rate=3e-4, n_steps=512, batch_size=64,
                n_epochs=10, gamma=0.99, ent_coef=0.01)
    model.learn(total_timesteps=50_000)

    # Evaluate
    obs, _ = env_v2g.reset()
    total_r = 0.0
    for _ in range(env_v2g.simulation_length):
        action, _ = model.predict(obs, deterministic=True)
        obs, r, done, trunc, _ = env_v2g.step(action)
        total_r += r
        if done or trunc:
            break
    print(f"PPO return (1 episode): {total_r:.2f}")

except ImportError:
    print("stable-baselines3 not installed.")
    print("Install with: pip install stable-baselines3")
    print()
    print("Illustrative command:")
    print("  from stable_baselines3 import PPO")
    print("  model = PPO('MlpPolicy', env_v2g, verbose=1)")
    print("  model.learn(total_timesteps=200_000)")

## Class summary

Over these four notebooks we have built the complete picture of reinforcement learning:

| Notebook | Topic | Key algorithms |
|---|---|---|
| 0 | V2G environment | ev2gym, heuristics |
| 1 | MDP, policies, value functions | Monte Carlo evaluation |
| 2 | Dynamic programming, value-based RL | Value iteration, DQN, SAC (intro) |
| 3 | Policy gradients | REINFORCE, A2C, PPO (intro) |

The V2G problem motivates all of this: it is a real-world continuous-action stochastic control problem that requires the full power of modern deep RL to solve efficiently.

**Going further:**
- SAC (off-policy actor-critic) → best for sample-efficient continuous control
- PPO (on-policy actor-critic with trust region) → robust, widely used
- Model-based RL → learn the transition model and plan inside it
- Multi-agent RL → multiple EVs / chargers with independent agents

And really many open topics to explore.